# Feature Engineering and Preprocessing

This notebook uses the Milestone 4 model-ready dataset. It creates a leakage-safe train/test split, then fits preprocessing exclusively on the training features. It intentionally does not fit a regression model.

In [1]:
from pathlib import Path

import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/processed/cardekho_model_ready.csv")
TARGET = "selling_price"
RANDOM_STATE = 42
TEST_SIZE = 0.20
BRAND_NORMALIZATION = {"ISUZU": "Isuzu"}

df = pd.read_csv(DATA_PATH)
brand_counts_before = df["brand"].value_counts()
df["brand"] = df["brand"].replace(BRAND_NORMALIZATION)
brand_normalization_audit = pd.DataFrame({
    "before": brand_counts_before.reindex(["ISUZU", "Isuzu"], fill_value=0),
    "after": df["brand"].value_counts().reindex(["ISUZU", "Isuzu"], fill_value=0),
})
assert df.shape == (15_235, 12)
assert TARGET in df.columns
assert (df["brand"] == "ISUZU").sum() == 0
assert (df["brand"] == "Isuzu").sum() == 9
display(brand_normalization_audit)
df.shape

,before,after
brand,,
ISUZU,2,0
Isuzu,7,9


(15235, 12)

## Canonicalize known brand labels

Before splitting or encoding, the deterministic mapping `ISUZU -> Isuzu` merges the two observed spellings (2 and 7 records) into one brand category. This is a feature-quality correction based only on the input label; it does not use `selling_price`.

## Separate the target before preprocessing

`selling_price` remains separate from the feature matrix. It is not encoded, scaled, or used to learn any preprocessing parameters. Target transformation is deliberately deferred to model experimentation.

In [2]:
X = df.drop(columns=TARGET)
y = df[TARGET].copy()

numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

assert set(numerical_features).isdisjoint(categorical_features)
assert set(numerical_features) | set(categorical_features) == set(X.columns)
assert TARGET not in X.columns

feature_groups = pd.DataFrame({
    "feature": numerical_features + categorical_features,
    "group": ["numerical"] * len(numerical_features) + ["categorical"] * len(categorical_features),
})
feature_groups

,feature,group
0,vehicle_age,numerical
1,km_driven,numerical
2,mileage,numerical
3,engine,numerical
4,max_power,numerical
5,seats,numerical
6,brand,categorical
7,model,categorical
8,seller_type,categorical
9,fuel_type,categorical


## Split before fitting

The holdout split is created before any imputer, scaler, or encoder is fitted. `random_state=42` makes the split reproducible. The target is not stratified because it is continuous; it remains untouched for a later regression evaluation milestone.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

assert X_train.index.intersection(X_test.index).empty
assert len(X_train) == 12_188
assert len(X_test) == 3_047
split_summary = pd.Series({
    "X_train_rows": len(X_train),
    "X_test_rows": len(X_test),
    "y_train_rows": len(y_train),
    "y_test_rows": len(y_test),
}, name="count")
split_summary.to_frame()

,count
X_train_rows,12188
X_test_rows,3047
y_train_rows,12188
y_test_rows,3047


## Define and fit the preprocessing pipeline

Numerical features receive median imputation and standard scaling. Categorical features receive most-frequent imputation followed by one-hot encoding. `handle_unknown="ignore"` makes prediction-time transformation safe when a new brand or model appears; its unseen category produces zeros in that encoded group instead of an error. All learned steps are fitted only with `X_train`.

In [4]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=0.30,
    verbose_feature_names_out=False,
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

assert X_train_transformed.shape[0] == len(X_train)
assert X_test_transformed.shape[0] == len(X_test)
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)

## Inspect transformed features

The transformed matrix contains the six scaled numerical columns followed by one-hot columns learned from the training categories. It is sparse because each categorical row activates only one value per categorical feature. The test set uses exactly the training-fitted feature layout.

In [5]:
transformed_feature_names = preprocessor.get_feature_names_out()
is_sparse_output = sparse.issparse(X_train_transformed)
nonzero_entries = X_train_transformed.nnz if is_sparse_output else (X_train_transformed != 0).sum()
transformed_density = nonzero_entries / (X_train_transformed.shape[0] * X_train_transformed.shape[1])
one_hot_encoder = preprocessor.named_transformers_["categorical"].named_steps["one_hot"]
category_counts = pd.Series(
    [len(categories) for categories in one_hot_encoder.categories_],
    index=categorical_features,
    name="training_categories",
)

transformed_summary = pd.Series({
    "train_rows": X_train_transformed.shape[0],
    "test_rows": X_test_transformed.shape[0],
    "numeric_feature_count": len(numerical_features),
    "one_hot_feature_count": int(category_counts.sum()),
    "transformed_feature_count": X_train_transformed.shape[1],
    "is_sparse": is_sparse_output,
    "train_matrix_density": round(float(transformed_density), 6),
}, name="value")

feature_name_preview = pd.DataFrame({"transformed_feature": transformed_feature_names})
display(transformed_summary.to_frame())
display(category_counts.to_frame())
display(feature_name_preview.head(20))
assert X_train_transformed.shape[1] == len(transformed_feature_names)
assert X_train_transformed.shape[1] == len(numerical_features) + int(category_counts.sum())
assert X_train_transformed.shape[1] == X_test_transformed.shape[1]

,value
train_rows,12188
test_rows,3047
numeric_feature_count,6
one_hot_feature_count,158
transformed_feature_count,164
is_sparse,True
train_matrix_density,0.067073


,training_categories
brand,30
model,118
seller_type,3
fuel_type,5
transmission_type,2


,transformed_feature
0,vehicle_age
1,km_driven
2,mileage
3,engine
4,max_power
5,seats
6,brand_Audi
7,brand_BMW
8,brand_Bentley
9,brand_Datsun


## Verify unknown-category handling

This non-training check replaces one test value with an unseen brand. Transformation succeeds with unchanged matrix dimensions, confirming that the one-hot encoder will not fail on an unseen categorical value at prediction time.

In [6]:
unknown_category_check = X_test.head(1).copy()
unknown_category_check.loc[:, "brand"] = "__unseen_brand__"
unknown_category_transformed = preprocessor.transform(unknown_category_check)

assert unknown_category_transformed.shape == (1, X_train_transformed.shape[1])
print("Unknown-category transformation completed successfully.")

Unknown-category transformation completed successfully.
